In [ ]:
import os
import sys
import gc
import subprocess
import logging
from glob import glob
from os.path import join

import numpy as np
import pandas as pd
import geopandas as gpd

import rasterio
import rioxarray as rxr
import xarray as xr
import zarr

# Terrain / elevation processing tools
import richdem as rd
import elevation

import matplotlib.pyplot as plt

from shapely.geometry import mapping
from shapely import wkt
import warnings
import pyproj

# Suppress warnings for cleaner batch processing output
warnings.filterwarnings("ignore")

# Ensure PROJ data path is explicitly set for CRS operations
pyproj.datadir.set_data_dir("/opt/conda/envs/macrosystems/share/proj")

In [ ]:
# Base data directory for reburn analysis outputs and inputs
data_path = os.path.join("/home", "jovyan", "work", "reburn_data")

# Final processed reburn dataset from first pipeline stage
final_path = os.path.join(
    data_path,
    "firedpy_severities_overlay",
    "firedpy_severities_overlay.gpkg"
)

# Intermediate clipped fire geometries (used for filling missing data)
clipped_path = os.path.join(
    data_path,
    "clipped_firedpy",
    "clipped_firedpy.gpkg"
)

# CBI raster stored in Zarr format for fast chunked access
zarr_path = os.path.join(data_path, "cbi_zarr", "cbi.zarr")

# GRIDMET climate data directory (used for gap-filling / recomputation)
gridmet_path = os.path.join(data_path, "gridmet")

# NOTE: Compared to the first pipeline file, this stage focuses on
# post-processing and data completion rather than initial feature construction.

In [ ]:
# Load clipped fire geometries produced by the first pipeline stage
clipped = gpd.read_file(clipped_path)

# Reproject to WGS84 for consistency with downstream processing steps
clipped = clipped.to_crs(epsg=4326)

# NOTE: In contrast to the first pipeline, this file assumes the geometry
# is already preprocessed and only standardizes CRS for correction/filling steps.

In [ ]:
# Open a sample GRIDMET precipitation dataset (used to infer structure/metadata)
ds = xr.open_dataset(os.path.join(gridmet_path, "pr_2000.nc"))

In [ ]:
# Open the output from the previous file
final = gpd.read_file(final_path)

# Fix crs, assign it properly to 5070, which is what the coordinates were written out as in the last file
final = final.set_crs(epsg=5070, allow_override=True)

# Once properly assigned, crs can then be set to the needed CRS in order to process this data
final = final.to_crs(epsg=4326)

In [ ]:
# Remove rows where ecoregion assignment failed or was not available
final = final[final['ecoregion'] != "N/A"]

# Rebuild a consistent reburn identifier from fire pair IDs
final["reburn_id"] = (
    final["fire1_id"].astype(str) + "_" + final["fire2_id"].astype(str)
)

In [ ]:
# Identify rows with missing precipitation values in reburn region (indicator for incomplete climate extraction)
missing_data = final[final["reburn1_pr"].isna()]

# Loop through all columns and report total missing values per field
for column in final.columns:
    total_nan = final[column].isnull().sum()
    print(f"Number of NA values in column: {total_nan}")

# NOTE: This stage is diagnostic only and is used to quantify gaps
# introduced during post-processing/filling rather than initial extraction.

In [ ]:
def plot_sample_reburn_gridmet(gdf, main_gdf, data_folder, pad=0.05):
    """
    Visualize GRIDMET climate variables for a randomly sampled reburn event
    with missing GRIDMET data across two fire periods (reburn1 and reburn2).

    Inputs:
        gdf: geopandas.GeoDataFrame
            Reburn dataset containing geometry and year_1/year_2 fields
        main_gdf: geopandas.GeoDataFrame
            Original fire event dataset used to retrieve start/end dates
        data_folder: string
            Base folder containing GRIDMET NetCDF files
        pad: float
            Spatial padding added to bounding box for raster extraction (default=0.05)

    Returns:
        None
            Displays a 2x4 matplotlib panel of climate variables
    """

    # Mapping of GRIDMET file prefixes to variable names inside NetCDF
    VAR_MAP = {
        "pr": "precipitation_amount",
        "tmmn": "air_temperature",
        "tmmx": "air_temperature",
        "vpd": "mean_vapor_pressure_deficit",
    }

    # --- Randomly sample a single reburn event ---
    row = gdf.sample(1).iloc[0]

    year_1 = int(row["year_1"])
    year_2 = int(row["year_2"])

    fire1_id = row["fire1_id"]
    fire2_id = row["fire2_id"]

    # --- Retrieve fire event date ranges from original dataset ---
    # NOTE: This is a key difference from the first pipeline:
    # dates are re-queried dynamically rather than stored-only.
    fire1_dates = main_gdf.loc[
        main_gdf["merge_id"] == fire1_id,
        ["ig_date", "last_date"]
    ].iloc[0]

    fire2_dates = main_gdf.loc[
        main_gdf["merge_id"] == fire2_id,
        ["ig_date", "last_date"]
    ].iloc[0]

    start1 = pd.to_datetime(fire1_dates["ig_date"])
    end1   = pd.to_datetime(fire1_dates["last_date"])

    start2 = pd.to_datetime(fire2_dates["ig_date"])
    end2   = pd.to_datetime(fire2_dates["last_date"])

    # --- Reproject geometry to WGS84 for GRIDMET compatibility ---
    geom = (
        gpd.GeoSeries([row.geometry], crs=gdf.crs)
        .to_crs(epsg=4326)
        .iloc[0]
    )

    minx, miny, maxx, maxy = geom.bounds

    # Create padded bounding box for raster subsetting
    lon_slice = slice(minx - pad, maxx + pad)
    lat_slice = slice(maxy + pad, miny - pad)

    results = {}

    # ---- Loop over both fire periods ----
    for period, year, start, end in zip(
        ["reburn1", "reburn2"],
        [year_1, year_2],
        [start1, start2],
        [end1, end2],
    ):

        for var, nc_var in VAR_MAP.items():

            # Open yearly GRIDMET dataset for variable
            ds = xr.open_dataset(
                join(data_folder, "gridmet", f"{var}_{year}.nc")
            )

            # Spatial + temporal subset
            ds_sub = ds.sel(
                lon=lon_slice,
                lat=lat_slice,
                day=slice(start, end),
            )

            # Compute mean over time dimension
            mean_da = ds_sub[nc_var].mean("day")

            # Store result for plotting
            results[f"{period}_{var}"] = mean_da

            ds.close()

    # -------- Plot 8-panel diagnostic figure --------
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()

    for ax, (name, da) in zip(axes, results.items()):
        da.plot(ax=ax, cmap="viridis")

        # Overlay fire geometry boundary for spatial context
        gpd.GeoSeries([geom]).boundary.plot(
            ax=ax,
            edgecolor="red",
            linewidth=2,
        )

        ax.set_title(name)
        ax.set_axis_off()

    plt.tight_layout()
    plt.show()


def fill_missing_climate(gdf, main_gdf, data_folder):
    """
    Fill missing GRIDMET-derived climate variables in a reburn dataset.

    This function re-queries GRIDMET using original fire geometries
    when values are missing from the processed dataset.

    Inputs:
        gdf: geopandas.GeoDataFrame
            Reburn dataset with missing climate variables
        main_gdf: geopandas.GeoDataFrame
            Original fire event dataset with geometry + dates
        data_folder: string
            Base directory containing GRIDMET data

    Returns:
        gdf: geopandas.GeoDataFrame
            Updated GeoDataFrame with missing values filled

    Note:
    Compared to the first pipeline:
        - This stage explicitly RECOMPUTES missing values instead of computing once
        - Is designed as a correction/repair layer rather than initial feature extraction
    """

    # Column prefixes corresponding to different spatial contexts
    prefixes = [
        "fire1_", "fire2_",
        "reburn1_", "reburn2_",
        "non_reburn_"
    ]

    # Iterate through reburn dataset row-by-row
    for idx, row in gdf.iterrows():

        # Identify missing climate-related columns for this row
        missing_cols = [
            col for col in gdf.columns
            if any(col.startswith(p) for p in prefixes)
            and pd.isna(row[col])
        ]

        # Skip fully complete rows
        if not missing_cols:
            continue

        # Progress logging (coarse, periodic)
        if idx % 50 == 0:
            print(f"On row {idx} of like 1500 or smth idk bro.")

        # Retrieve fire identifiers
        fire1_id = row["fire1_id"]
        fire2_id = row["fire2_id"]

        # Lookup original fire metadata from main dataset
        fire1_row = main_gdf.loc[main_gdf["merge_id"] == fire1_id].iloc[0]
        fire2_row = main_gdf.loc[main_gdf["merge_id"] == fire2_id].iloc[0]

        fire1_geom = fire1_row.geometry
        fire2_geom = fire2_row.geometry

        fire1_start = pd.to_datetime(fire1_row["ig_date"])
        fire1_end   = pd.to_datetime(fire1_row["last_date"])

        fire2_start = pd.to_datetime(fire2_row["ig_date"])
        fire2_end   = pd.to_datetime(fire2_row["last_date"])

        # Fill each missing column individually
        for col in missing_cols:

            # ---- Fire 1 climate context ----
            if col.startswith("fire1_"):
                geom = fire1_geom
                start = fire1_start
                end   = fire1_end
                year  = start.year

            # ---- Fire 2 climate context ----
            elif col.startswith("fire2_"):
                geom = fire2_geom
                start = fire2_start
                end   = fire2_end
                year  = start.year

            # ---- Reburn overlap (fire1-year context) ----
            elif col.startswith("reburn1_"):
                geom = row.geometry
                start = fire1_start
                end   = fire1_end
                year  = start.year

            # ---- Reburn overlap (fire2-year context) ----
            elif col.startswith("reburn2_"):
                geom = row.geometry
                start = fire2_start
                end   = fire2_end
                year  = start.year

            # ---- Non-reburn region (difference geometry) ----
            elif col.startswith("non_reburn_"):
                geom = fire2_geom.difference(fire1_geom)
                start = fire2_start
                end   = fire2_end
                year  = start.year

            else:
                continue

            # Recompute GRIDMET means for missing field
            climate_vals = compute_gridmet_mean(
                geom, start, end, year, data_folder
            )

            # Extract variable name (pr, tmmn, etc.)
            var = col.split("_")[-1]

            # Fill missing value if available
            if var in climate_vals:
                gdf.at[idx, col] = climate_vals[var]

    return gdf


def calc_av_slope_elevation(geom):
    """
    Compute average elevation, slope, and aspect for a given geometry using a temporary DEM clip.
    This is being done in this notebook instead of the previous due to issues it was causing with 
    the notebook crashing. 

    Inputs:
        geom: shapely geometry
            Input geometry used to define bounding box for elevation extraction

    Returns:
        tuple
            (average_elevation, average_slope, average_aspect)
            Returns NaNs if computation fails
    """

    try:
        # Temporary file used for clipped DEM extraction
        path = join(data_path, "temp_elevation.tif")

        # Clip global elevation dataset to geometry bounding box
        elevation.clip(geom.bounds, output=path)

        # Load clipped DEM into richdem format
        dem = rd.LoadGDAL(path)

        # Compute terrain derivatives
        slope = rd.TerrainAttribute(dem, attrib='slope_riserun')
        aspect = rd.TerrainAttribute(dem, attrib='aspect')

        # Clean up temporary raster file
        os.remove(path)

        # Free elevation cache and Python memory
        elevation.clean()
        gc.collect()

        # Compute mean terrain metrics
        av_elev = np.nanmean(dem)
        av_slope = np.nanmean(slope)
        av_aspect = np.nanmean(aspect)

        return (av_elev, av_slope, av_aspect)

    except subprocess.CalledProcessError:
        # Fail-safe return if clipping or processing fails
        return (np.nan, np.nan, np.nan)


In [ ]:
# Plot used to help diagnose the issue with the GRIDMET missing data
plot_sample_reburn_gridmet(missing_data, clipped, data_path, 0.02)

In [ ]:
# Run post-processing step to fill missing GRIDMET-derived climate values
filled = fill_missing_climate(final, clipped, data_path)

# Remove any remaining rows with missing values (final quality filter)
filled = filled.dropna()

# Sort reburn events by spatial extent (largest first for prioritization/analysis)
filled = filled.sort_values(by='reburn_area', ascending=False)

In [ ]:
# Initialize storage for terrain metrics (fire1, fire2, reburn/overlay, and difference geometry)
fire1_ele = []
fire1_slope = []
fire1_aspect = []

fire2_ele = []
fire2_slope = []
fire2_aspect = []

overlay_ele = []
overlay_slope = []
overlay_aspect = []

diff_ele = []
diff_slope = []
diff_aspect = []

# Geometry storage (re-attaching spatial objects after feature computation)
fire1_geometry = []
fire2_geometry = []
diff_geometry = []

# Iterate over fully processed reburn dataset
for idx, row in filled.iterrows():

    # Progress logging every 50 rows
    if idx % 50 == 0:
        print(f"On row {idx}")

    # Identify fire pair
    fire1_id = row["fire1_id"]
    fire2_id = row["fire2_id"]

    # Lookup original clipped geometries for each fire event
    fire1_row = clipped.loc[clipped["merge_id"] == fire1_id].iloc[0]
    fire2_row = clipped.loc[clipped["merge_id"] == fire2_id].iloc[0]

    fire1_geom = fire1_row.geometry
    fire2_geom = fire2_row.geometry

    # Reburn (overlay) geometry already stored in dataset
    overlay_geom = row.geometry

    # Compute non-overlap region (difference of fire2 minus fire1)
    diff_geom = fire2_geom.difference(fire1_geom)

    # Compute terrain metrics for each spatial component
    fire1_vars = calc_av_slope_elevation(fire1_geom)
    fire2_vars = calc_av_slope_elevation(fire2_geom)
    overlay_vars = calc_av_slope_elevation(overlay_geom)
    diff_vars = calc_av_slope_elevation(diff_geom)

    # ---- Fire 1 terrain features ----
    fire1_ele.append(fire1_vars[0])
    fire1_slope.append(fire1_vars[1])
    fire1_aspect.append(fire1_vars[2])

    # ---- Fire 2 terrain features ----
    fire2_ele.append(fire2_vars[0])
    fire2_slope.append(fire2_vars[1])
    fire2_aspect.append(fire2_vars[2])

    # ---- Reburn (intersection) terrain features ----
    overlay_ele.append(overlay_vars[0])
    overlay_slope.append(overlay_vars[1])
    overlay_aspect.append(overlay_vars[2])

    # ---- Non-reburn (difference) terrain features ----
    diff_ele.append(diff_vars[0])
    diff_slope.append(diff_vars[1])
    diff_aspect.append(diff_vars[2])

    # Store geometry objects for downstream spatial analysis
    fire1_geometry.append(fire1_geom)
    fire2_geometry.append(fire2_geom)
    diff_geometry.append(diff_geom)

# Attach computed terrain variables back onto dataframe
filled['fire1_ele'] = fire1_ele
filled['fire1_slope'] = fire1_slope
filled['fire1_aspect'] = fire1_aspect

filled['fire2_ele'] = fire2_ele
filled['fire2_slope'] = fire2_slope
filled['fire2_aspect'] = fire2_aspect

filled['reburn_ele'] = overlay_ele
filled['reburn_slope'] = overlay_slope
filled['reburn_aspect'] = overlay_aspect

filled['non_reburn_ele'] = diff_ele
filled['non_reburn_slope'] = diff_slope
filled['non_reburn_aspect'] = diff_aspect

filled['fire1_geometry'] = fire1_geometry
filled['fire2_geometry'] = fire2_geometry
filled['non_reburn_geometry'] = diff_geometry

In [ ]:
# Check the data has correctly filled the dataset
for column in filled.columns:
    total_nan = filled[column].isnull().sum()
    print(f"Number of NA values in column: {total_nan}")

In [ ]:
# Output data
filled_path = os.path.join(os.getcwd(), "filled_data", "filled_overlay.csv")
filled.to_csv(filled_path, index=False)
filled = pd.read_csv(filled_path)

In [ ]:
# Output data with only one geometry column in order to make it a geopackage
filled_gdf  = filled.drop(['fire1_geometry', 'fire2_geometry', 'non_reburn_geometry'], axis=1)
filled_gdf = gpd.GeoDataFrame(filled_gdf, geometry = filled.geometry.apply(wkt.loads), crs = final.crs)
filled_gdf_path = os.path.join(os.getcwd(), "filled_data", "filled_overlay.gpkg")
filled_gdf.to_file(filled_gdf_path, index=False)